# Thompson Sampling

## The Same Problem, A Different Philosophy

We are solving the same **multi-armed bandit problem** from the UCB notebook: 10 ads, 10,000 users, unknown click-through rates. We want to find the best ad while wasting as little budget as possible on bad ones.

UCB approached this deterministically: compute a formula, pick the highest number. **Thompson Sampling** takes a fundamentally different, probabilistic approach:

> Instead of computing a confidence bound, maintain a **probability distribution** over each ad's true click rate, and sample from it to make each decision.

---

## The Bayesian Intuition

Thompson Sampling is a **Bayesian algorithm**. Here is the core idea:

1. Start with a **prior belief** that each ad has an unknown click rate somewhere between 0 and 1 (uniform distribution — all equally plausible).
2. Each time you show an ad and observe a click (reward = 1) or no-click (reward = 0), **update your belief** about that ad.
3. When choosing which ad to show next, **sample a number** from each ad's current belief distribution. Show the ad with the highest sampled value.
4. Repeat.

Over time, the belief distributions for bad ads tighten around low values and the best ad's distribution tightens around its true click rate — making it almost always win the sampling step.

---

## The Beta Distribution: Perfect for Click Rates

Click rates are probabilities between 0 and 1. The **Beta distribution** is the natural choice for modelling a probability:

$$\text{Beta}(\alpha, \beta)$$

| Parameter | Meaning in our context |
|-----------|------------------------|
| $\alpha$ | Number of clicks observed (successes) + 1 |
| $\beta$ | Number of non-clicks observed (failures) + 1 |

Adding 1 to both is the **Laplace smoothing** that gives the uniform prior (Beta(1,1)) before any data. As clicks accumulate, $\alpha$ grows and the distribution shifts right (towards higher click rates). As no-clicks accumulate, $\beta$ grows and it shifts left.

---

## UCB vs Thompson Sampling

| | UCB | Thompson Sampling |
|-|-----|-------------------|
| **Approach** | Deterministic formula | Probabilistic sampling |
| **Exploration** | Explicit confidence bonus | Implicit through sampling variance |
| **Computation** | Simpler math | Requires sampling from Beta distribution |
| **Empirical performance** | Strong | Often better in practice |
| **Theoretical guarantees** | Logarithmic regret | Bayesian optimal (Lai-Robbins bound) |

**In practice**, Thompson Sampling often converges faster and is more robust because natural uncertainty in the distribution handles exploration automatically — no tuning needed.

---

## What We Will Build

1. Maintain Beta distribution parameters for all 10 ads
2. At each round: sample from each ad's Beta distribution, show the ad with the highest sample
3. Update the distribution based on the observed reward
4. Visualise which ad the algorithm converges on after 10,000 rounds

## Step 1: Import Libraries

| Library | Why we need it |
|---------|---------------|
| `numpy` | Array operations |
| `matplotlib` | Visualising the selection histogram |
| `pandas` | Loading the CSV dataset |
| `random` | `random.betavariate(alpha, beta)` — samples from the Beta distribution |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Step 2: Load the Dataset

Same dataset as the UCB notebook: `ads_ctr_optimization.csv` with 10,000 rows and 10 columns.

Each row is a user. Each column is one ad. The value (0 or 1) indicates whether that user *would have* clicked that ad. In a real system you only observe the reward for the ad you chose — the rest are counterfactual.

**This is a simulated environment**, which is standard practice for testing bandit algorithms before deploying them.

In [ ]:
dataset = pd.read_csv('Ads_CTR_Optimisation.csv')

## Step 3: Implement Thompson Sampling

We track two counters per ad:

| Variable | Meaning |
|----------|---------|
| `numbers_of_rewards_1[i]` | How many times ad $i$ was clicked ($\alpha - 1$) |
| `numbers_of_rewards_0[i]` | How many times ad $i$ was not clicked ($\beta - 1$) |

**The algorithm at each round:**

1. For each ad $i$, draw a **random sample** from `Beta(rewards_1[i] + 1, rewards_0[i] + 1)`
   - Early on: both parameters are 1, so Beta(1,1) is uniform — any click rate is equally likely
   - After 50 clicks and 10 non-clicks: Beta(51, 11) peaks near 0.82 — strong evidence of a good ad
2. Select the ad with the **highest sampled value** — this is stochastic, not deterministic
3. Observe the reward
4. Increment `rewards_1` if clicked, `rewards_0` if not

**Why does sampling work for exploration?** An ad with only 3 observations has high variance in its Beta distribution — it can easily sample a high value by chance, giving it another trial. An ad with 500 observations has a tight distribution — it only wins the sampling step if its true rate is genuinely high. Exploration is automatic and diminishes naturally as evidence accumulates.

In [ ]:
import random
N = 10000
d = 10
ads_selected = []
numbers_of_rewards_1 = [0] * d
numbers_of_rewards_0 = [0] * d
total_reward = 0
for n in range(0, N):
    ad = 0
    max_random = 0
    for i in range(0, d):
        random_beta = random.betavariate(numbers_of_rewards_1[i] + 1, numbers_of_rewards_0[i] + 1)
        if random_beta > max_random:
            max_random = random_beta
            ad = i
    ads_selected.append(ad)
    reward = dataset.values[n, ad]
    if reward == 1:
        numbers_of_rewards_1[ad] = numbers_of_rewards_1[ad] + 1
    else:
        numbers_of_rewards_0[ad] = numbers_of_rewards_0[ad] + 1
    total_reward = total_reward + reward

## Step 4: Visualise the Results

The histogram shows how many of the 10,000 rounds each ad was selected.

**What you should observe:**

- **One bar dominates** — the ad with the highest true click rate gets selected the vast majority of the time
- **All other bars are short but present** — Thompson Sampling explored all ads early, then committed to the best
- The convergence should be even sharper than UCB, because sampling variance decreases faster for well-tested ads

**Comparing to UCB:** Both algorithms converge to the same best ad. Thompson Sampling tends to waste fewer rounds on bad ads because it naturally gives them less exploration budget once evidence against them accumulates.

**Real-world deployment note:** Thompson Sampling is used in production at companies including Google, Microsoft, and LinkedIn for personalised recommendations and ad selection — precisely because it handles the exploration-exploitation tradeoff automatically without manual tuning of exploration parameters.

In [ ]:
plt.hist(ads_selected)
plt.title('Histogram of ads selections')
plt.xlabel('Ads')
plt.ylabel('Number of times each ad was selected')
plt.show()